<a href="https://colab.research.google.com/github/mariaaslam706/CVD_data_XQFD_demo/blob/main/CVD_XQFD_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#All Libraries
!pip install explainerdashboard
!pip install graphviz

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from explainerdashboard import RegressionExplainer, ExplainerDashboard

!pip install dalex
import dalex as dx

import pandas as pd
import numpy as np

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

import warnings
warnings.filterwarnings('ignore')

! pip install plotly
import plotly
plotly.offline.init_notebook_mode()
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'colab'

In [ ]:
#Load data

df = pd.read_csv(r"/content/heart_failure_2.csv")
print (df.shape)
print (df.columns.values)

#count values in target column
df.head()

In [ ]:
#Seperate the labels
X= df.drop(['DEATH_EVENT'], axis=1)
Y = df['DEATH_EVENT']

In [ ]:
#train and test data
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)
print(X_train.shape,Y_train.shape,X_test.shape,Y_test.shape)

In [ ]:
#Training the model
model = RandomForestRegressor(n_estimators=500, random_state = 42)
model.fit(X_train, Y_train.values.ravel())

In [ ]:
#######Dashboard XAI#######
explainer = RegressionExplainer(model, X_test, Y_test)
#Start the Dashboard
db = ExplainerDashboard(explainer,whatif=False)
#Running the app on a local port 3050
db.run(port=8050)

In [ ]:
ExplainerDashboard.terminate(3050)

In [ ]:
#######Dalex XAI#######
exp = dx.Explainer(model,X_train,Y_train)
#Model performance metrix
exp.model_performance()

In [ ]:
exp.model_parts()

In [ ]:
exp.model_parts().plot()

In [ ]:
data_sample = X_test.iloc[7]
data_sample

In [ ]:
Y_test.iloc[7]
print("RFR:",model.predict([data_sample]))

In [ ]:
data_frame = pd.DataFrame(data_sample).T

In [ ]:
explanation = exp.predict_parts(data_frame)
explanation.result

In [ ]:
explanation.plot()

In [ ]:
john = pd.DataFrame({'age': [55],
                       'anaemia': [1],
                       'creatinine_phosphokinase': [850],
                       'diabetes': [1],
                       'ejection_fraction': [72],
                       'high_blood_pressure': [1],
                       'platelets': 22000 ,
                       'serum_creatinine': [1.3] ,
                       'serum_sodium': [172] ,
                       'sex': [1] ,
                       'smoking':[0]},
                      index = ['John'])

In [ ]:
exp.predict(X)[0:10]

In [ ]:
exp.predict(john)

In [ ]:
Maria = pd.DataFrame({'age': [20],
                       'anaemia': [0],
                       'creatinine_phosphokinase': [0],
                       'diabetes': [0],
                       'ejection_fraction': [30],
                       'high_blood_pressure': [0],
                       'platelets': 0 ,
                       'serum_creatinine': [14] ,
                       'serum_sodium': [0] ,
                       'sex': [1] ,
                       'smoking':[1]},
                      index = ['Maria'])

In [ ]:
exp.predict(X)[0:10]

In [ ]:
exp.predict(Maria)

In [ ]:
bd_john = exp.predict_parts(john, type='break_down', label=john.index[0])
bd_interactions_john = exp.predict_parts(john, type='break_down_interactions', label="John+")

sh_Maria = exp.predict_parts(Maria, type='shap', B=10, label=Maria.index[0])

In [ ]:
bd_john.result

In [ ]:
bd_john.plot(bd_interactions_john)

In [ ]:
sh_Maria.result.loc[sh_Maria.result.B == 0, ]

In [ ]:
sh_Maria.plot(bar_width=16)

In [ ]:
exp.predict_parts(john, type='shap', B=10, label=john.index[0]).plot(max_vars=5)

In [ ]:
mp = exp.model_performance(model_type = 'regression')
mp.result

In [ ]:
cp_Maria = exp.predict_profile(Maria, label=Maria.index[0])
cp_john = exp.predict_profile(john, label=john.index[0])

cp_Maria.result.head()